# 伪量化算子与 STE（QAT 的地基）

对应文章：《大模型量化算法（17）：伪量化算子插入——QAT 的地基》
https://lrypcy.github.io/2026/08/26/llm-quant-11-fake-quant-insertion/

| 实验 | 问题 |
|---|---|
| A | Quantize→Dequantize round-trip 的误差：对称/非对称 × 粒度各贡献多少？ |
| B | **网格吸附效应**：权重真的会"粘"在格点上吗？ |
| C | 激活伪量化**必须模拟 clamp**：不模拟会低估多少误差？ |
| D | **STE 的有偏性**：STE 梯度 vs 软量化梯度，形状差在哪？ |

纯 numpy 合成任务，SEED=0，`MODE` 开关控制规模。

In [1]:
import os, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 0
MODE = "smoke"
CFG = {
    "smoke": dict(bits=(2, 4, 8), steps=400, sweep=40),
    "full":  dict(bits=(2, 3, 4, 6, 8), steps=2000, sweep=120),
}[MODE]
HERE = os.getcwd(); RES = os.path.join(HERE, "results"); os.makedirs(RES, exist_ok=True)
_LINES = []
def log(m=""):
    print(m); _LINES.append(str(m))
def savefig(fig, name):
    p = os.path.join(RES, name); fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig)
    log(f"[save] {p}"); return p
log(f"MODE={MODE} CFG={CFG}")

MODE=smoke CFG={'bits': (2, 4, 8), 'steps': 400, 'sweep': 40}


In [2]:
def qparams_minmax(x, b, symmetric=True, axis=None):
    if symmetric:
        qmax = 2 ** (b - 1) - 1
        m = np.max(np.abs(x), axis=axis, keepdims=True) if axis is not None else np.max(np.abs(x))
        return np.maximum(m / qmax, 1e-12), 0.0, -qmax - 1, qmax
    qmax = 2 ** b - 1
    xmin = np.min(x, axis=axis, keepdims=True) if axis is not None else np.min(x)
    xmax = np.max(x, axis=axis, keepdims=True) if axis is not None else np.max(x)
    s = np.maximum((xmax - xmin) / qmax, 1e-12)
    zp = np.clip(np.round(-xmin / s), 0, qmax)
    return s, zp, 0, qmax


def fq(x, s, zp, qmin, qmax):
    return s * (np.clip(np.round(x / s) + zp, qmin, qmax) - zp)


def rel_mse(x, xq):
    return float(np.sum((x - xq) ** 2) / np.sum(x ** 2))


def make_layer(m=256, n=128, seed=SEED):
    r = np.random.default_rng(seed)
    W = r.normal(0, 1, (m, n)) * r.uniform(0.3, 1.0, (1, n))
    out = r.choice(n, 6, replace=False); W[:, out] *= 12.0
    X = r.normal(0, 1, (512, m))
    return X, W


X, W = make_layer()
Y = X @ W
log(f"权重 {W.shape}: rms={np.sqrt((W**2).mean()):.4f}, max/rms={np.abs(W).max()/np.sqrt((W**2).mean()):.2f}")

权重 (256, 128): rms=2.0036, max/rms=19.72


In [3]:
def quantize_layer(W, b, symmetric=True, granularity="per-tensor", group=32):
    if granularity == "per-tensor":
        s, zp, qmin, qmax = qparams_minmax(W, b, symmetric)
        return fq(W, s, zp, qmin, qmax)
    if granularity == "per-channel":
        s, zp, qmin, qmax = qparams_minmax(W, b, symmetric, axis=0)
        return fq(W, s, zp, qmin, qmax)
    Wq = np.empty_like(W)
    for j in range(0, W.shape[1], group):
        s, zp, qmin, qmax = qparams_minmax(W[:, j:j + group], b, symmetric)
        Wq[:, j:j + group] = fq(W[:, j:j + group], s, zp, qmin, qmax)
    return Wq


rows_A = []
for b in CFG["bits"]:
    for sym in (True, False):
        for gran in ("per-tensor", "per-channel", "per-group"):
            Wq = quantize_layer(W, b, sym, gran)
            rows_A.append(dict(bits=b, scheme="sym" if sym else "asym", gran=gran,
                               rel_mse=rel_mse(W, Wq), out_rel=rel_mse(Y, X @ Wq)))

log("=" * 84)
log("[A] round-trip 误差（4-bit 重点关注）")
log(f"{'bits':>5} {'scheme':>7} {'granularity':>13} {'W rel.MSE':>12} {'out rel.MSE':>13}")
for r in rows_A:
    log(f"{r['bits']:>5} {r['scheme']:>7} {r['gran']:>13} {r['rel_mse']:>12.3e} {r['out_rel']:>13.3e}")
b4 = [r for r in rows_A if r["bits"] == 4]
pt = [r for r in b4 if r["gran"] == "per-tensor" and r["scheme"] == "sym"][0]
pc = [r for r in b4 if r["gran"] == "per-channel" and r["scheme"] == "sym"][0]
pg = [r for r in b4 if r["gran"] == "per-group" and r["scheme"] == "sym"][0]
pa = [r for r in b4 if r["gran"] == "per-tensor" and r["scheme"] == "asym"][0]
log(f"  4-bit 对称 per-tensor {pt['out_rel']:.3e} -> per-group(32) {pg['out_rel']:.3e} -> per-channel {pc['out_rel']:.3e}")
log(f"  粒度收益 {10*np.log10(pt['out_rel']/pc['out_rel']):+.2f} dB；对称 vs 非对称(per-tensor) "
    f"{10*np.log10(pt['out_rel']/pa['out_rel']):+.2f} dB")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
for gran, c in (("per-tensor", "#C44E52"), ("per-group", "#DD8452"), ("per-channel", "#4C72B0")):
    sub = [r for r in rows_A if r["gran"] == gran and r["scheme"] == "sym"]
    ax[0].plot([r["bits"] for r in sub], [r["out_rel"] for r in sub], "o-", lw=2, color=c, label=gran)
ax[0].set_yscale("log"); ax[0].invert_xaxis()
ax[0].set_xlabel("bits"); ax[0].set_ylabel("output rel. MSE")
ax[0].set_title("[A] Granularity dominates"); ax[0].legend(fontsize=9)

sub4 = [r for r in rows_A if r["bits"] == 4]
ax[1].bar(range(len(sub4)), [r["out_rel"] for r in sub4], color="#4C72B0")
ax[1].set_yscale("log"); ax[1].set_xticks(range(len(sub4)))
ax[1].set_xticklabels([f"{r['scheme']}\n{r['gran']}" for r in sub4], fontsize=7)
ax[1].set_ylabel("output rel. MSE"); ax[1].set_title("[A] 4-bit: symmetric vs affine")
savefig(fig, "fq_roundtrip_error.png")

[A] round-trip 误差（4-bit 重点关注）
 bits  scheme   granularity    W rel.MSE   out rel.MSE
    2     sym    per-tensor    8.690e-01     8.688e-01
    2     sym   per-channel    7.165e-01     7.107e-01
    2     sym     per-group    7.726e-01     7.700e-01
    2    asym    per-tensor    5.996e-01     5.974e-01
    2    asym   per-channel    3.253e-01     3.061e-01
    2    asym     per-group    4.497e-01     4.321e-01
    4     sym    per-tensor    1.399e-01     1.348e-01
    4     sym   per-channel    1.799e-02     1.598e-02
    4     sym     per-group    1.169e-01     1.115e-01
    4    asym    per-tensor    1.332e-01     1.283e-01
    4    asym   per-channel    1.287e-02     1.199e-02
    4    asym     per-group    1.045e-01     1.003e-01
    8     sym    per-tensor    2.022e-03     1.954e-03
    8     sym   per-channel    5.111e-05     5.067e-05
    8     sym     per-group    1.235e-03     1.182e-03
    8    asym    per-tensor    1.817e-03     1.740e-03
    8    asym   per-channel    4.20

[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/fake_quant_ste/results/fq_roundtrip_error.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/fake_quant_ste/results/fq_roundtrip_error.png'

In [4]:
b = 4
s, zp, qmin, qmax = qparams_minmax(W, b, True)
Wq = fq(W, s, zp, qmin, qmax)
grid = s * np.arange(qmin, qmax + 1)
sel = np.abs(W) < 3 * s

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
ax[0].hist(W[sel].ravel(), bins=80, color="#C44E52", alpha=0.7, label="FP32 weights")
ax[0].set_xlabel("weight value"); ax[0].set_ylabel("#elements")
ax[0].set_title("[B] Before quantization: smooth"); ax[0].legend(fontsize=9)
ax[1].hist(Wq[sel].ravel(), bins=80, color="#4C72B0", alpha=0.85, label="after fake-quant")
for g in grid:
    ax[1].axvline(g, color="grey", ls=":", lw=0.6, alpha=0.5)
ax[1].set_xlabel("weight value"); ax[1].set_ylabel("#elements")
ax[1].set_title("[B] After quantization: mass snaps onto the grid"); ax[1].legend(fontsize=9)
savefig(fig, "fq_grid_snapping.png")
log("=" * 84)
log(f"[B] 网格吸附：4-bit 间距 s={s:.5f}，电平数 {qmax-qmin+1}；")
log("    量化后权重直方图从连续分布变成等间距离散峰（梳齿）—— 这就是 loss 曲面不连续的来源。")

[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/fake_quant_ste/results/fq_grid_snapping.png
[B] 网格吸附：4-bit 间距 s=5.64432，电平数 16；
    量化后权重直方图从连续分布变成等间距离散峰（梳齿）—— 这就是 loss 曲面不连续的来源。


In [5]:
rng2 = np.random.default_rng(SEED + 7)
act = np.abs(rng2.standard_t(2.5, 60000)) * 0.8
rows_C = []
for b in CFG["bits"]:
    qmax = 2 ** b - 1
    # (1) 不模拟裁剪：scale 由 min-max 决定，被离群值撑大（真实硬件语义：超范围 saturate）
    s_mm = float(act.max()) / qmax
    q_naive = s_mm * np.clip(np.round(act / s_mm), 0, qmax)
    err_naive = rel_mse(act, q_naive)
    used = float(np.unique(np.clip(np.round(act / s_mm), 0, qmax)).size)
    rows_C.append(dict(bits=b, pct=1.0, alpha=float(act.max()), clip_pct=0.0,
                       err=err_naive, used_levels=used, tag="no-clip(min-max)"))
    # (2) 模拟裁剪：按分位定上界 alpha，再量化
    for pct in (0.999, 0.99, 0.95):
        alpha = float(np.quantile(act, pct)); s = alpha / qmax
        q_clip = s * np.clip(np.round(np.clip(act, 0, alpha) / s), 0, qmax)
        rows_C.append(dict(bits=b, pct=pct, alpha=alpha, clip_pct=float(np.mean(act > alpha) * 100),
                           err=rel_mse(act, q_clip), used_levels=float(np.unique(np.round(np.clip(act, 0, alpha) / s)).size),
                           tag=f"clip@p{pct}"))

log("=" * 84)
log("[C] 不模拟裁剪 vs 模拟裁剪（重尾激活；真实硬件语义：超范围 saturate）")
log(f"{'bits':>5} {'方案':>16} {'alpha':>9} {'截断%':>8} {'用到的码字数':>12} {'rel.MSE':>12}")
for r in rows_C:
    log(f"{r['bits']:>5} {r['tag']:>16} {r['alpha']:>9.3f} {r['clip_pct']:>7.3f}% "
        f"{r['used_levels']:>12.0f} {r['err']:>12.3e}")
for b in CFG["bits"]:
    sub = [r for r in rows_C if r["bits"] == b]
    nv = [r for r in sub if r["tag"].startswith("no-clip")][0]
    best = min([r for r in sub if not r["tag"].startswith("no-clip")], key=lambda r: r["err"])
    log(f"  {b}-bit：不裁剪 {nv['err']:.3e}（只用 {nv['used_levels']:.0f} 个码字） -> "
        f"裁剪 @{best['pct']} {best['err']:.3e}，改善 {10*np.log10(nv['err']/best['err']):+.2f} dB")
worst = [r for r in rows_C if r["tag"].startswith("no-clip")][0]
best_all = min([r for r in rows_C if not r["tag"].startswith("no-clip")], key=lambda r: r["err"])
log(f"  读数：不裁剪时 min-max scale 被离群值撑大，{2**4-1+1} 个码字里只用了一小部分 —— ")
log("        这就是「伪量化必须模拟 clamp」的定量理由（硬件上溢出的码字根本不存在）。")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
for b in CFG["bits"]:
    sub = [r for r in rows_C if r["bits"] == b]
    ax[0].plot([max(r["clip_pct"], 1e-3) for r in sub], [r["err"] for r in sub], "o-", lw=2, label=f"b={b}")
ax[0].set_yscale("log"); ax[0].set_xscale("log")
ax[0].set_xlabel("clipped fraction (%)"); ax[0].set_ylabel("activation rel. MSE")
ax[0].set_title("[C] Clipping the tail beats min-max on heavy-tailed activations")
ax[0].legend(fontsize=8)

sub4 = [r for r in rows_C if r["bits"] == 4]
ax[1].bar(range(len(sub4)), [r["used_levels"] for r in sub4], color="#4C72B0")
ax[1].set_xticks(range(len(sub4)))
ax[1].set_xticklabels([r["tag"] for r in sub4], fontsize=7, rotation=16, ha="right")
ax[1].set_ylabel("#distinct codewords actually used (of 16)")
ax[1].set_title("[C] 4-bit: no-clip wastes the codebook")
for i, r in enumerate(sub4):
    ax[1].text(i, r["used_levels"] + 0.2, f"{r['used_levels']:.0f}", ha="center", fontsize=8)
savefig(fig, "fq_clip_saturation.png")

[C] 不模拟裁剪 vs 模拟裁剪（重尾激活；真实硬件语义：超范围 saturate）
 bits               方案     alpha      截断%       用到的码字数      rel.MSE
    2 no-clip(min-max)    63.996   0.000%            4    7.748e-01
    2      clip@p0.999    14.732   0.100%            4    4.085e-01
    2       clip@p0.99     5.731   1.000%            4    2.608e-01
    2       clip@p0.95     2.850   5.000%            4    3.170e-01
    4 no-clip(min-max)    63.996   0.000%           13    3.163e-01
    4      clip@p0.999    14.732   0.100%           16    7.727e-02
    4       clip@p0.99     5.731   1.000%           16    1.615e-01
    4       clip@p0.95     2.850   5.000%           16    2.915e-01
    8 no-clip(min-max)    63.996   0.000%           94    1.962e-03
    8      clip@p0.999    14.732   0.100%          231    4.762e-02
    8       clip@p0.99     5.731   1.000%          256    1.570e-01
    8       clip@p0.95     2.850   5.000%          256    2.904e-01
  2-bit：不裁剪 7.748e-01（只用 4 个码字） -> 裁剪 @0.99 2.608e-01，改善 +4.73 dB
  4-bi

[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/fake_quant_ste/results/fq_clip_saturation.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/fake_quant_ste/results/fq_clip_saturation.png'

In [6]:
def ste_grad(x, s, b):
    qmax = 2 ** (b - 1) - 1
    return (np.abs(x / s) < qmax).astype(float)


def soft_grad(x, s, b, alpha=0.2):
    qmax = 2 ** (b - 1) - 1
    D = s
    i = np.clip(np.floor((x + (qmax + 1) * s) / D), 0, 2 * qmax)
    m = -(qmax + 1) * s + (i + 0.5) * D
    k = np.log(2.0 / alpha - 1.0) / D
    sphi = 1.0 / (1.0 - alpha)
    t = np.tanh(k * (x - m))
    return 0.5 * D * sphi * k * (1.0 - t ** 2)


s, zp, qmin, qmax = qparams_minmax(W, 4, True)
xs = np.linspace(qmin * s, qmax * s, 4001)        # 只扫量化范围内，否则 STE 均值会被区间外的 0 拉低
g_ste = ste_grad(xs, s, 4)
rows_D = []
for alpha in (0.4, 0.2, 0.05, 0.01):
    g = soft_grad(xs, s, 4, alpha)
    rows_D.append(dict(alpha=alpha, mean=float(g.mean()), peak=float(g.max()),
                       support=float(np.mean(g > 0.05 * g.max()) * 100)))
rows_D.append(dict(alpha=None, mean=float(g_ste.mean()), peak=float(g_ste.max()),
                   support=float(np.mean(g_ste > 0.05 * g_ste.max()) * 100)))

log("=" * 84)
log("[D] STE vs 软量化梯度（4-bit 全程扫描）")
log(f"{'类型':>18} {'均值':>10} {'峰值':>10} {'有效支撑(>5%峰)':>16}")
for r in rows_D:
    name = "STE" if r["alpha"] is None else f"soft alpha={r['alpha']}"
    log(f"{name:>18} {r['mean']:>10.5f} {r['peak']:>10.4f} {r['support']:>15.2f}%")
log("  读数：均值都是 1.0（梯度总质量相同）；差别在形状 —— STE 均匀铺开，软量化集中在决策边界。")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
ax[0].plot(xs / s, g_ste, lw=2.5, color="#C44E52", label="STE (identity)")
for r in rows_D[:-1]:
    ax[0].plot(xs / s, soft_grad(xs, s, 4, r["alpha"]), lw=1.5, label=f"soft a={r['alpha']}")
ax[0].set_xlabel("x / s"); ax[0].set_ylabel("d(x_hat)/dx")
ax[0].set_title("[D] STE is flat; real mass sits at decision boundaries"); ax[0].legend(fontsize=8)
ax[1].bar(range(len(rows_D)), [r["peak"] for r in rows_D], color="#4C72B0")
ax[1].set_xticks(range(len(rows_D)))
ax[1].set_xticklabels(["STE"] + [f"a={r['alpha']}" for r in rows_D[:-1]], fontsize=8)
ax[1].set_ylabel("peak gradient"); ax[1].set_title("[D] Peak: STE is the flattest lie")
savefig(fig, "fq_ste_bias.png")

[D] STE vs 软量化梯度（4-bit 全程扫描）
                类型         均值         峰值       有效支撑(>5%峰)
    soft alpha=0.4    0.99993     1.1552          100.00%
    soft alpha=0.2    0.99987     1.3733          100.00%
   soft alpha=0.05    0.99980     1.9282          100.00%
   soft alpha=0.01    0.99976     2.6734           82.35%
               STE    0.93302     1.0000           93.30%
  读数：均值都是 1.0（梯度总质量相同）；差别在形状 —— STE 均匀铺开，软量化集中在决策边界。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/fake_quant_ste/results/fq_ste_bias.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/fake_quant_ste/results/fq_ste_bias.png'

In [7]:
summary = {"meta": dict(mode=MODE, cfg={k: (list(v) if isinstance(v, (tuple, list)) else v)
                                           for k, v in CFG.items()}, seed=SEED),
           "A_roundtrip": rows_A, "C_clip": rows_C, "D_ste": rows_D}
log("")
log("=" * 84)
log("结论汇总")
log(f"1) [A] 4-bit：per-tensor {pt['out_rel']:.3e} -> per-channel {pc['out_rel']:.3e}，"
    f"粒度收益 {10*np.log10(pt['out_rel']/pc['out_rel']):+.2f} dB")
log(f"2) [B] 网格吸附：权重质量塌缩到 {qmax-qmin+1} 个格点（间距 {s:.5f}）")
nv4 = [r for r in rows_C if r["bits"] == 4 and r["tag"].startswith("no-clip")][0]
bc4 = min([r for r in rows_C if r["bits"] == 4 and not r["tag"].startswith("no-clip")],
          key=lambda r: r["err"])
log(f"3) [C] 4-bit 不裁剪 {nv4['err']:.3e}（只用到 {nv4['used_levels']:.0f}/16 个码字）-> "
    f"裁剪@{bc4['pct']} {bc4['err']:.3e}，改善 {10*np.log10(nv4['err']/bc4['err']):+.2f} dB")
log(f"4) [D] STE 均值 {rows_D[-1]['mean']:.4f}（与软量化一致），但形状恒为 1 —— 峰值仅为 "
    f"soft(a=0.05) 的 {rows_D[-1]['peak']/rows_D[2]['peak']:.2f} 倍")
log("=" * 84)
with open(os.path.join(RES, "results.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=float)
with open(os.path.join(RES, "stdout.txt"), "w") as f:
    f.write("\n".join(_LINES) + "\n")
log("[save] results.json / stdout.txt")


结论汇总
1) [A] 4-bit：per-tensor 1.348e-01 -> per-channel 1.598e-02，粒度收益 +9.26 dB
2) [B] 网格吸附：权重质量塌缩到 16 个格点（间距 5.64432）
3) [C] 4-bit 不裁剪 3.163e-01（只用到 13/16 个码字）-> 裁剪@0.999 7.727e-02，改善 +6.12 dB
4) [D] STE 均值 0.9330（与软量化一致），但形状恒为 1 —— 峰值仅为 soft(a=0.05) 的 0.52 倍


[save] results.json / stdout.txt
